In [1]:
import sys
import os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))

import numpy as np
import pandas as pd
import json
from src.data_process import GenerateAnomaly, PrepareDf
from src.metric import ProbTimeDetection, FalseRate


In [2]:
from canari import (
    DataProcess,
    Model,
    Optimizer,
    SKF,
    plot_data,
    plot_prediction,
    plot_skf_states,
    plot_states,
)
from canari.component import LocalTrend, LocalAcceleration, LstmNetwork, WhiteNoise

/opt/miniconda3/envs/bm/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-05-01 16:18:42,497	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
2026-05-01 16:18:42,610	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [3]:
# Data folder 
data_folder = "/Users/vuongdai/GitHub/bm/detrend_data/monthly/"

# Data file
data_file = "ts_monthly_values_standardize.csv"

# Time file
data_file_time = "ts_monthly_datetimes.csv"

sensor_names = pd.read_csv(data_folder + data_file, nrows=0, delimiter=",").columns.tolist()
df = pd.read_csv(data_folder + data_file, skiprows=1, delimiter=",", header=None)
df_time = pd.read_csv(data_folder + data_file_time, skiprows=1, delimiter=",", header=None)

df_dict = PrepareDf(df, df_time)

# Anomaly info.
anomaly_info_file = "../detrend_data/anomaly_info/anomaly_info.json"
with open(anomaly_info_file, "r") as f:
    anomaly_info = json.load(f)

In [4]:
# Generate anomalies
df_with_anomaly = GenerateAnomaly(data=df_dict, anomaly_info=anomaly_info, col=[0,1])

In [5]:
df_with_anomaly[0]

{'level': {'0.45': {0:                value
   2012-01-01 -0.724202
   2012-02-01 -0.539961
   2012-03-01 -0.133944
   2012-04-01 -0.073183
   2012-05-01  0.286713
   ...              ...
   2022-12-01 -0.105230
   2023-01-01 -0.038757
   2023-02-01  0.168649
   2023-03-01  0.181104
   2023-04-01  0.842460
   
   [136 rows x 1 columns],
   1:                value
   2012-01-01 -0.724202
   2012-02-01 -0.539961
   2012-03-01 -0.133944
   2012-04-01 -0.523183
   2012-05-01  0.286713
   ...              ...
   2022-12-01 -0.105230
   2023-01-01 -0.038757
   2023-02-01  0.168649
   2023-03-01  0.181104
   2023-04-01  0.842460
   
   [136 rows x 1 columns],
   2:                value
   2012-01-01 -0.724202
   2012-02-01 -0.539961
   2012-03-01 -0.133944
   2012-04-01 -0.523183
   2012-05-01 -0.163287
   ...              ...
   2022-12-01 -0.105230
   2023-01-01 -0.038757
   2023-02-01  0.168649
   2023-03-01  0.181104
   2023-04-01  0.842460
   
   [136 rows x 1 columns],
   3:            

In [6]:
# SKF model
sigma_v = 5e-2
local_trend = LocalTrend()
local_acceleration = LocalAcceleration()
lstm_network = LstmNetwork(
    look_back_len=19,
    num_features=1,
    num_layer=1,
    num_hidden_unit=50,
    manual_seed=1,
    smoother=False,
)
noise = WhiteNoise(std_error=sigma_v)

# Normal model
model = Model(
    local_trend,
    lstm_network,
    noise,
)

#  Abnormal model
ab_model = Model(
    local_acceleration,
    lstm_network,
    noise,
)

output_col = [0]
df_temp = df.iloc[:,[1]]
data_processor = DataProcess(
    data=df_temp,
    # time_covariates=["hour_of_day"],
    train_split=0.4,
    validation_split=0.1,
    output_col=output_col,
)
train_data, validation_data, test_data, all_data = data_processor.get_splits()

# Switching Kalman filter
skf = SKF(
    norm_model=model,
    abnorm_model=ab_model,
    std_transition_error=1e-4,
    norm_to_abnorm_prob=1e-5,
)
skf.auto_initialize_baseline_states(train_data["y"][0:24])

for epoch in range(1):
    skf.lstm_train(
        train_data=train_data,
        validation_data=validation_data,
    )

In [7]:
def detect_synthetic_anomaly(
    data,
    detector,
    threshold: float = 0.1,
):
    anomaly_score = {}
    for ts in data:
        anomaly_score[ts] = {}
        for anomaly_type in data[ts]:
            anomaly_score[ts][anomaly_type] = {}
            for i in data[ts][anomaly_type]:
                anomaly_score[ts][anomaly_type][i] = {}
                for j in data[ts][anomaly_type][i]:
                    _data = {}
                    _data["y"] = data[ts][anomaly_type][i][j].values
                    _data["x"] = [[] for _ in range(len(_data["y"]))]
                    _data["time"] = data[ts][anomaly_type][i][j].index

                    _anomaly_score, _ = detector.filter(data=_data)
                    
                    _anomaly_score = _anomaly_score > threshold
                    _df_score_temp = pd.DataFrame({"value": _anomaly_score}, index=_data["time"])
                    anomaly_score[ts][anomaly_type][i][j] = _df_score_temp

    return anomaly_score

In [8]:
def detect_anomaly(
    data,
    detector,
    threshold: float = 0.1,
):
    anomaly_score = {}
    for ts in data:
        _df = data[ts] 
        _data = {}
        _data["y"] = _df.values
        _data["x"] = [[] for _ in range(len(_data["y"]))]
        _data["time"] = _df.index
        _anomaly_score, _ = detector.filter(data=_data)
        _anomaly_score = _anomaly_score > threshold
        anomaly_score[ts] = pd.DataFrame({"value": _anomaly_score}, index=_data["time"])

    return anomaly_score

In [9]:
# Estimate anomaly score
anomaly_score_synthetic = detect_synthetic_anomaly(detector=skf, data=df_with_anomaly)
anomaly_score_test_set = detect_anomaly(detector=skf, data=df_dict)



In [10]:
prob_detection, time_to_detection = ProbTimeDetection(anomaly_score_synthetic, anomaly_info)


In [11]:
prob_detection

{0: {'level': {'0.45': 1.0, '0.01': 1.0, '0.1': 1.0},
  'trend': {'0.01': 1.0, '0.05': 0.4, '0.1': 0.4}},
 1: {'level': {'0.45': 1.0, '0.01': 1.0, '0.1': 1.0},
  'trend': {'0.01': 1.0, '0.05': 1.0, '0.1': 0.5}}}

In [12]:
time_to_detection

{0: {'level': {'0.45': Timedelta('297 days 04:48:00'),
   '0.01': Timedelta('215 days 14:24:00'),
   '0.1': Timedelta('185 days 02:24:00')},
  'trend': {'0.01': Timedelta('197 days 07:12:00'),
   '0.05': Timedelta('0 days 00:00:00'),
   '0.1': Timedelta('0 days 00:00:00')}},
 1: {'level': {'0.45': Timedelta('258 days 21:36:00'),
   '0.01': Timedelta('136 days 09:36:00'),
   '0.1': Timedelta('133 days 14:24:00')},
  'trend': {'0.01': Timedelta('136 days 09:36:00'),
   '0.05': Timedelta('256 days 02:24:00'),
   '0.1': Timedelta('9 days 00:00:00')}}}

In [13]:
false_rate = FalseRate(anomaly_score_test_set)

In [14]:
false_rate

{0: 0.8891187925998053,
 1: 1.511501947419669,
 2: 2.642362546506821,
 3: 1.158494037046435,
 4: 1.0001369112814895,
 5: 1.4365326059375791,
 6: 0.18535904592742958,
 7: 0.40901455767077266,
 8: 1.207937164117404,
 9: 0.37071809185485916,
 10: 0.7414361837097183,
 11: 0.8396551724137932,
 12: 1.7780933062880326,
 13: 1.531135902636917,
 14: 1.158494037046435,
 15: 0.4170578533367165,
 16: 0.16666666666666666,
 17: 1.1666666666666667,
 18: 1.0861299915038232,
 19: 0.6024168992641461,
 20: 0.09267952296371479,
 21: 2.689037529179386,
 22: 2.1477862331373228,
 23: 1.3473340040241448,
 24: 0.09998631261976458,
 25: 1.010722933241093,
 26: 0.40336830480397573,
 27: 0.3999452504790583,
 28: 2.9012988407041647,
 29: 1.0529982702287142,
 30: 1.9618483412322276,
 31: 0.8743743199129489,
 32: 1.2000328551089694,
 33: 0.794885745375408,
 34: 0.0,
 35: 0.3333333333333333,
 36: 0.8218946894689468,
 37: 0.5316593886462883,
 38: 0.9114160948222083,
 39: 0.3038053649407361,
 40: 0.8333333333333334,
 4

In [15]:
import numpy as np
np.random.seed(42)
population = np.round(np.arange(0, 0.8, 0.001), 3)  # [0.00, 0.01, ..., 0.79]
values = sorted(np.random.choice(population, size=50, replace=False).tolist())
print(values)

[0.023, 0.03, 0.039, 0.063, 0.065, 0.066, 0.067, 0.078, 0.139, 0.174, 0.215, 0.244, 0.25, 0.26, 0.286, 0.323, 0.327, 0.346, 0.36, 0.377, 0.383, 0.395, 0.398, 0.423, 0.456, 0.49, 0.525, 0.526, 0.529, 0.533, 0.534, 0.57, 0.578, 0.595, 0.596, 0.604, 0.61, 0.621, 0.622, 0.635, 0.637, 0.658, 0.667, 0.692, 0.696, 0.721, 0.741, 0.746, 0.76, 0.796]
